# Interaktivní průzkum vybraných algoritmů strojového učení ve scikit-learn

Notebook umožňuje vytvářet syntetická data, měnit jejich velikost a míru šumu a sledovat výsledek modelu.

Zaměříme se na **KNN**, **DTC** (klasifikační rozhodovací strom), **MLP** (vícevrstvou neuronovou síť) a **K-Means** (shlukování bez známých tříd).


## 1. Klasifikace: KNN, DTC a MLP

U klasifikace známe pro trénovací vzorky jejich třídu. Model se učí rozlišit třídy a vytváří rozhodovací hranici.

* **KNN (`n_neighbors`)** určuje třídu bodu podle jeho nejbližších sousedů.
* **DTC (`max_depth`)** postupně dělí prostor hodnot; velká hloubka může znamenat přeučení.
* **MLP (`hidden_layer_sizes`)** je neuronová síť; počet neuronů určuje její kapacitu.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import make_classification, make_circles, make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42


### Interaktivní klasifikace

Měňte typ dat, šum, algoritmus a jeho hyperparametr. Barva pozadí ukazuje třídu předpovězenou modelem.


In [ ]:
def vytvor_klasifikacni_data(pocet_vzorku, sum, typ_dat):
    if typ_dat == "Lineárně oddělitelné třídy":
        return make_classification(n_samples=pocet_vzorku, n_features=2, n_redundant=0,
            n_informative=2, n_clusters_per_class=1, flip_y=sum, class_sep=1.2,
            random_state=RANDOM_STATE)
    if typ_dat == "Dva půlměsíce":
        return make_moons(n_samples=pocet_vzorku, noise=sum, random_state=RANDOM_STATE)
    return make_circles(n_samples=pocet_vzorku, noise=sum, factor=0.5, random_state=RANDOM_STATE)


def vykresli_klasifikaci(pocet_vzorku, sum, typ_dat, algoritmus, k_sousedu, max_hloubka, neurony_mlp):
    X, y = vytvor_klasifikacni_data(pocet_vzorku, sum, typ_dat)
    X = StandardScaler().fit_transform(X)
    if algoritmus == "KNN":
        model, parametr = KNeighborsClassifier(n_neighbors=k_sousedu), f"počet sousedů = {k_sousedu}"
    elif algoritmus == "DTC (rozhodovací strom)":
        model, parametr = DecisionTreeClassifier(max_depth=max_hloubka, random_state=RANDOM_STATE), f"maximální hloubka = {max_hloubka}"
    else:
        model = MLPClassifier(hidden_layer_sizes=(neurony_mlp,), max_iter=1500, random_state=RANDOM_STATE)
        parametr = f"neurony ve skryté vrstvě = {neurony_mlp}"
    model.fit(X, y)
    x_min, x_max, y_min, y_max = X[:,0].min()-0.5, X[:,0].max()+0.5, X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.28, cmap="coolwarm")
    plt.scatter(X[:,0], X[:,1], c=y, edgecolor="black", cmap="coolwarm", s=35)
    plt.xlabel("Příznak 1 (standardizovaný)")
    plt.ylabel("Příznak 2 (standardizovaný)")
    plt.title(f"{algoritmus}: {parametr} | přesnost na trénovacích datech = {model.score(X, y):.2f}")
    plt.show()


In [ ]:
typ_klas_dat = widgets.Dropdown(options=["Lineárně oddělitelné třídy", "Dva půlměsíce", "Souosé kružnice"], value="Dva půlměsíce", description="Typ dat:")
pocet_vzorku = widgets.IntSlider(min=50, max=500, step=10, value=200, description="Počet bodů:")
sum = widgets.FloatSlider(min=0, max=0.5, step=0.02, value=0.20, description="Šum:")
algoritmus = widgets.Dropdown(options=["KNN", "DTC (rozhodovací strom)", "MLP (neuronová síť)"], value="KNN", description="Algoritmus:")
k_sousedu = widgets.IntSlider(min=1, max=25, value=5, description="K sousedů:")
max_hloubka = widgets.IntSlider(min=1, max=12, value=4, description="Max. hloubka:")
neurony_mlp = widgets.IntSlider(min=2, max=100, step=2, value=20, description="Neurony MLP:")

rozhrani_klas = widgets.VBox([typ_klas_dat, pocet_vzorku, sum, algoritmus, k_sousedu, max_hloubka, neurony_mlp])
vystup_klas = widgets.interactive_output(vykresli_klasifikaci, {
    "pocet_vzorku": pocet_vzorku, "sum": sum, "typ_dat": typ_klas_dat, "algoritmus": algoritmus,
    "k_sousedu": k_sousedu, "max_hloubka": max_hloubka, "neurony_mlp": neurony_mlp})
display(rozhrani_klas, vystup_klas)


## 2. Shlukování: K-Means

K-Means je metoda **učení bez učitele**: předem neznáme třídy bodů. Určíme počet shluků (*K*) a algoritmus opakovaně přiřazuje body k nejbližšímu středu shluku a přepočítává jeho polohu. Křížky označují nalezené středy shluků.


In [ ]:
def vykresli_kmeans(pocet_bodu, sum_km, typ_dat_km, pocet_shluku):
    if typ_dat_km == "Oddělené shluky":
        X = make_classification(n_samples=pocet_bodu, n_features=2, n_redundant=0, n_informative=2,
            n_clusters_per_class=1, class_sep=1.7, random_state=RANDOM_STATE)[0]
    elif typ_dat_km == "Dva půlměsíce":
        X = make_moons(n_samples=pocet_bodu, noise=sum_km, random_state=RANDOM_STATE)[0]
    else:
        X = make_circles(n_samples=pocet_bodu, noise=sum_km, factor=0.5, random_state=RANDOM_STATE)[0]
    X = StandardScaler().fit_transform(X)
    model = KMeans(n_clusters=pocet_shluku, n_init=20, random_state=RANDOM_STATE)
    stitky = model.fit_predict(X)
    plt.figure(figsize=(7, 5))
    plt.scatter(X[:,0], X[:,1], c=stitky, cmap="tab10", edgecolor="black", s=35)
    plt.scatter(model.cluster_centers_[:,0], model.cluster_centers_[:,1], marker="X", s=220,
        c="black", edgecolor="white", linewidth=1.5, label="Středy shluků")
    plt.xlabel("Příznak 1 (standardizovaný)")
    plt.ylabel("Příznak 2 (standardizovaný)")
    plt.title(f"K-Means: K = {pocet_shluku}, inerciální kritérium = {model.inertia_:.1f}")
    plt.legend()
    plt.show()


In [ ]:
typ_km_dat = widgets.Dropdown(options=["Oddělené shluky", "Dva půlměsíce", "Souosé kružnice"], value="Oddělené shluky", description="Typ dat:")
pocet_bodu_km = widgets.IntSlider(min=50, max=500, step=10, value=200, description="Počet bodů:")
sum_km = widgets.FloatSlider(min=0, max=0.5, step=0.02, value=0.10, description="Šum:")
pocet_shluku = widgets.IntSlider(min=2, max=6, value=2, description="Počet shluků:")
rozhrani_km = widgets.VBox([typ_km_dat, pocet_bodu_km, sum_km, pocet_shluku])
vystup_km = widgets.interactive_output(vykresli_kmeans, {
    "pocet_bodu": pocet_bodu_km, "sum_km": sum_km, "typ_dat_km": typ_km_dat, "pocet_shluku": pocet_shluku})
display(rozhrani_km, vystup_km)


### Otázky k interpretaci

1. Který klasifikátor dokáže na datech typu „dva půlměsíce“ vytvořit vhodnou hranici?
2. Kdy je rozhodovací strom příliš složitý vzhledem k datům?
3. Proč K-Means nedělí půlměsíce a souosé kružnice intuitivně, i když na grafu vidíme zjevné skupiny?
4. Jaká informace chybí K-Means ve srovnání s klasifikátory?
